# SpikingBrain-7B — QLoRA Fine-Tuning

Fine-tuning con **quantizzazione int4 + adattatori LoRA** (QLoRA).  
Solo ~0.5% dei parametri viene addestrato, riducendo la VRAM da ~56 GB (full FT) a ~12-14 GB.

| | Full Fine-Tuning | **QLoRA (questo notebook)** |
|---|---|---|
| Parametri trainabili | 7B (100%) | ~35M (0.5%) |
| VRAM richiesta | ~56 GB | **~12-14 GB** |
| GPU consigliata | A100 80GB | **A100 40GB** |
| Qualita' risultato | Massima | Molto buona |

> **Prerequisito**: Runtime GPU A100.  
> `Runtime → Change runtime type → A100 GPU`  
> (richiede Colab Pro)

## 1. Verifica GPU

In [ ]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

if not torch.cuda.is_available():
    raise RuntimeError("GPU non trovata! Abilita il runtime A100.")

gpu_name  = torch.cuda.get_device_name(0)
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU   : {gpu_name}")
print(f"VRAM  : {vram_gb:.1f} GB")

if vram_gb < 14:
    raise RuntimeError(
        f"VRAM insufficiente ({vram_gb:.1f} GB). QLoRA richiede almeno 14 GB."
    )
print("OK: VRAM sufficiente per QLoRA.")

## 2. Installazione Dipendenze

> ~10 minuti la prima volta (compilazione flash_attn).

In [ ]:
# Librerie base
!pip install -q --upgrade pip
!pip install -q transformers==4.55.2
!pip install -q flash-linear-attention==0.1
!pip install -q flash-attn==2.7.3 --no-build-isolation

# QLoRA stack
!pip install -q bitsandbytes>=0.43.0         # quantizzazione int4/int8
!pip install -q peft>=0.11.0                 # LoRA adapters
!pip install -q trl>=0.8.6                   # SFTTrainer
!pip install -q accelerate>=0.30.0           # training multi-GPU / mixed precision

# Dataset e utility
!pip install -q datasets>=2.19.0
!pip install -q modelscope
!pip install -q scipy pyyaml decorator einops

print("Installazione completata!")

## 3. Clona Repository e Scarica Modello Base

In [ ]:
import os, sys

REPO_DIR  = "/content/SpikingBrain-7B"
MODEL_DIR = "/content/spikingbrain_7b_base"
MODEL_ID  = "Panyuqi/V1-7B-base"   # modello base (pre-trained, non SFT)

# 1. Codice del modello
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/BICLab/SpikingBrain-7B.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, f"{REPO_DIR}/hf_7B_model")
print(f"Codice modello: {REPO_DIR}/hf_7B_model")

# 2. Pesi del modello (ModelScope)
from modelscope import snapshot_download

if not os.path.exists(MODEL_DIR):
    print(f"Download '{MODEL_ID}' (~14 GB)...")
    model_path = snapshot_download(MODEL_ID, cache_dir=MODEL_DIR)
else:
    import glob
    dirs = glob.glob(f"{MODEL_DIR}/**", recursive=False)
    model_path = dirs[0] if dirs else MODEL_DIR
    print(f"Modello gia' presente: {model_path}")

print(f"Modello pronto: {model_path}")

## 4. Caricamento in 4-bit (BitsAndBytes)

Il modello viene quantizzato **al volo in int4** durante il caricamento.  
I calcoli avvengono comunque in bfloat16 (double quantization per risparmiare ulteriore memoria).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configurazione quantizzazione int4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4: ottimale per LLM
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,      # double quantization: -0.4 GB extra
)

print("Caricamento tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    padding_side='right',   # right-padding per training (diverso dall'inferenza)
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Caricamento modello in int4...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False          # obbligatorio per training
model.config.pretraining_tp = 1

mem_gb = torch.cuda.memory_allocated() / 1e9
print(f"Modello caricato! VRAM usata: {mem_gb:.1f} GB")
print(f"Parametri totali: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

## 5. Configurazione LoRA

LoRA aggiunge matrici di basso rango **A** e **B** ai layer lineari selezionati.  
Solo questi pesi vengono addestrati; il modello base rimane congelato.

```
Architettura SpikingBrain-7B (28 layer):
  Layer pari  (0,2,4,...,26) → GatedLinearAttention: q_proj, k_proj, v_proj, o_proj
  Layer dispari (1,3,5,...,27) → FlashAttention:       q_proj, k_proj, v_proj, o_proj
  Tutti i layer              → GLU (FFN):              gate_proj, up_proj, down_proj
```

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepara il modello quantizzato per il training
# (abilita gradient checkpointing + cast layer norms in float32)
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,

    # Rango della decomposizione (r): trade-off qualita'/memoria
    # r=8  → ~18M parametri trainabili, veloce
    # r=16 → ~35M parametri trainabili, qualita' maggiore
    # r=32 → ~70M parametri trainabili, vicino al full FT
    r=16,
    lora_alpha=32,       # scaling: lora_alpha / r = 2.0
    lora_dropout=0.05,
    bias="none",

    # Layer lineari (nn.Linear) presenti in entrambi i tipi di attention + FFN
    # NOTA: gk_proj e' nn.Sequential -> non supportato da PEFT, escluso
    target_modules=[
        # FlashAttention (layer dispari: 1,3,5,...,27)
        "q_proj", "k_proj", "v_proj", "o_proj",
        # GLU / FFN (tutti i layer)
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Preparazione Dataset

Di default usa **Alpaca** (52K esempi instruction-following in inglese).  
Sostituisci `DATASET_ID` e `formatting_func` con i tuoi dati.

In [ ]:
from datasets import load_dataset

# ----------------------------------------------------------------
# PERSONALIZZA QUI: sostituisci con il tuo dataset
# ----------------------------------------------------------------
DATASET_ID  = "tatsu-lab/alpaca"   # HuggingFace Hub dataset ID
DATASET_SPLIT = "train"            # split da usare
MAX_SAMPLES = 5000                 # None = tutto il dataset
MAX_SEQ_LEN = 1024                 # lunghezza massima sequenza (token)
# ----------------------------------------------------------------

dataset = load_dataset(DATASET_ID, split=DATASET_SPLIT)
if MAX_SAMPLES:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

print(f"Dataset: {DATASET_ID}")
print(f"Esempi:  {len(dataset)}")
print(f"Colonne: {dataset.column_names}")
print("\nEsempio:")
print(dataset[0])

In [ ]:
# ----------------------------------------------------------------
# Funzione di formattazione: adatta al formato del tuo dataset
# Questo esempio usa il formato Alpaca (instruction + input + output)
# ----------------------------------------------------------------
def formatting_func(example):
    """
    Converte un esempio del dataset nel formato chat di SpikingBrain.
    Il tokenizer usa il chat template Qwen2 con tag <|im_start|> / <|im_end|>.
    """
    instruction = example.get("instruction", "").strip()
    inp         = example.get("input", "").strip()
    output      = example.get("output", "").strip()

    # Costruisce il prompt utente
    user_content = instruction
    if inp:
        user_content = f"{instruction}\n\n### Input:\n{inp}"

    messages = [
        {"role": "system",    "content": "You are a helpful assistant."},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": output},
    ]

    # Applica il chat template del tokenizer
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # False per training (include risposta)
    )
    return text


# Verifica formattazione su un esempio
sample = formatting_func(dataset[0])
print("Esempio formattato:\n")
print(sample[:800], "...")

## 7. Training con SFTTrainer

`SFTTrainer` di TRL gestisce automaticamente:
- packing delle sequenze
- troncamento a `MAX_SEQ_LEN`
- gradient checkpointing
- logging loss

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/spikingbrain_qlora_output"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # --- Batch e ottimizzazione ---
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # batch effettivo = 2 * 4 = 8
    gradient_checkpointing=True,

    # --- Precisione ---
    bf16=True,
    fp16=False,
    tf32=True,                        # A100: abilita TensorFloat32

    # --- Learning rate ---
    optim="paged_adamw_32bit",        # ottimizzatore con paging per VRAM ridotta
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    # --- Epoche e step ---
    num_train_epochs=3,
    max_steps=-1,                     # -1 = usa num_train_epochs

    # --- Sequenze ---
    max_seq_length=MAX_SEQ_LEN,
    packing=True,                     # impacchetta sequenze brevi per efficienza

    # --- Logging e salvataggio ---
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",                 # cambia in "tensorboard" o "wandb" se vuoi

    # --- Altro ---
    dataset_text_field=None,          # usiamo formatting_func
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=formatting_func,
    args=training_args,
)

print("Trainer pronto. Avvio training...")
print(f"  Batch effettivo : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Epoche          : {training_args.num_train_epochs}")
print(f"  Max seq len     : {training_args.max_seq_length}")

In [ ]:
# Avvia il training
trainer.train()
print("Training completato!")

## 8. Salvataggio Adattatori LoRA

Salva **solo gli adattatori** (~70-140 MB), non il modello intero (~14 GB).

In [ ]:
ADAPTER_DIR = "/content/spikingbrain_lora_adapters"

# Salva adattatori LoRA
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adattatori salvati in: {ADAPTER_DIR}")

# Dimensione su disco
import subprocess
size = subprocess.run(['du', '-sh', ADAPTER_DIR], capture_output=True, text=True)
print(f"Dimensione: {size.stdout.split()[0]}")

In [ ]:
# Opzionale: comprimi e scarica su Google Drive
# !zip -r /content/lora_adapters.zip {ADAPTER_DIR}

# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/lora_adapters.zip /content/drive/MyDrive/

## 9. (Opzionale) Merge degli Adattatori nel Modello Base

Per ottenere un singolo modello senza dipendenza da PEFT,  
esegui il merge degli adattatori LoRA nei pesi originali.

> **Nota**: il merge richiede il modello in float16/bfloat16, **non in int4**.  
> Serve ~28 GB di VRAM — richiede A100 40GB.

In [ ]:
from peft import PeftModel

MERGED_DIR = "/content/spikingbrain_merged"

print("Caricamento modello base in bfloat16 per il merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print("Caricamento adattatori LoRA...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Merge adattatori nei pesi base...")
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Modello merged salvato in: {MERGED_DIR}")

## 10. Test del Modello Fine-Tunato

In [ ]:
# Testa il modello LoRA (senza merge)
model.eval()

def generate_qlora(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_ids = generated[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


# Testa con un esempio dello stesso tipo del dataset di training
test_prompt = "Explain the difference between supervised and unsupervised learning."
print(f"Prompt: {test_prompt}\n")
response = generate_qlora(test_prompt)
print("=" * 60)
print(response)
print("=" * 60)

## 11. Ricarica Adattatori in una Sessione Successiva

Per riprendere il lavoro o fare inferenza con gli adattatori gia' trainati:

In [ ]:
# --- CELLA STANDALONE: esegui questa in una sessione separata ---
#
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# import torch
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_path,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )
# model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
# tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
# model.eval()
print("Vedi commento nella cella per il codice di ricarica.")

---

## Riferimenti e Personalizzazione

### Parametri LoRA

| Parametro | Default | Note |
|-----------|---------|------|
| `r` | 16 | Rango: 8 (veloce) → 64 (qualita' alta) |
| `lora_alpha` | 32 | Scaling = alpha/r; tipicamente 2*r |
| `lora_dropout` | 0.05 | Regolarizzazione |
| `target_modules` | q,k,v,o + FFN | Aggiungi `"gk_proj"` se PEFT aggiunge supporto Sequential |

### Dataset Personalizzato

Per usare i tuoi dati, nella cella 6 puoi fare:
```python
from datasets import Dataset
data = [
    {"instruction": "...", "input": "", "output": "..."},
    ...
]
dataset = Dataset.from_list(data)
```

### Layer GLA e LoRA
I layer **GatedLinearAttention** (layer pari 0,2,4,...) hanno `q_proj`, `k_proj`, `v_proj`, `o_proj`  
che condividono lo stesso nome con i layer FlashAttention. PEFT li individua automaticamente  
grazie al nome del modulo, quindi entrambi i tipi di attention vengono adattati.